# Pipeline 02 — Pré-processamento (Median)

Processa todas as imagens de `02_dataset/images`, aplica operações globais configuráveis e o filtro **Median** definido em `00_common/02_filtering.ipynb`.

Saída: `results/` com sufixo `_PP_PL_2`.

## GLOBAL OPERATIONS CONFIGURATION

Alterar os valores abaixo para gerar variantes do dataset de treino (sem alterar geometria da imagem).

In [ ]:
# ==========================================================
# GLOBAL OPERATIONS CONFIGURATION
# ==========================================================

APPLY_BRIGHTNESS = False
BRIGHTNESS_OFFSET = 20

APPLY_CONTRAST = False
CONTRAST_FACTOR = 1.20

APPLY_GAMMA = False
GAMMA_VALUE = 1.10

PIPELINE_NUMBER = 2
PIPELINE_FILTER_NAME = "Median"

## Biblioteca comum (`00_common`)

Carrega as funções já implementadas nos notebooks partilhados — **sem reimplementar algoritmos**.

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.image as mpimg

%run ../00_common/00_generic.ipynb
%run ../00_common/01_global_operations.ipynb
%run ../00_common/02_filtering.ipynb

## Caminhos de entrada e saída

- **Entrada:** `02_dataset/images`
- **Saída:** `results/` (apenas nesta pasta)

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "02_dataset").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parents[2]

INPUT_DIR = PROJECT_ROOT / "02_dataset" / "images"
OUTPUT_DIR = Path("results").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTENSOES_VALIDAS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Entrada: {INPUT_DIR}")
print(f"Saída:   {OUTPUT_DIR}")

## Processamento em lote

Filtro desta pipeline: **Median**.

In [ ]:
def aplicar_operacoes_globais(imagem_uint8):
    imagem_processada = imagem_uint8
    if APPLY_CONTRAST or APPLY_BRIGHTNESS:
        alpha = CONTRAST_FACTOR if APPLY_CONTRAST else 1.0
        beta = BRIGHTNESS_OFFSET if APPLY_BRIGHTNESS else 0
        imagem_processada = manipulacao_brilho_contraste(
            imagem_processada, alpha=alpha, b=beta
        )
    if APPLY_GAMMA:
        imagem_processada = transformacao_gamma(imagem_processada, gamma=GAMMA_VALUE)
    return imagem_processada


def aplicar_filtro_pipeline(imagem_uint8):
    return filtro_median(imagem_uint8)


ficheiros_entrada = sorted(
    p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() in EXTENSOES_VALIDAS
)

if len(ficheiros_entrada) == 0:
    print(f"AVISO: nenhuma imagem encontrada em {INPUT_DIR}")
else:
    print(f"A processar {len(ficheiros_entrada)} imagem(ns)...")

for caminho_entrada in ficheiros_entrada:
    imagem_original = carregar_imagem(caminho_entrada)
    altura_original, largura_original = imagem_original.shape

    imagem_apos_global = aplicar_operacoes_globais(imagem_original)
    imagem_final = aplicar_filtro_pipeline(imagem_apos_global)

    altura_final, largura_final = imagem_final.shape
    if (altura_final, largura_final) != (altura_original, largura_original):
        raise ValueError(
            f"Geometria alterada em {caminho_entrada.name}: "
            f"{altura_original}x{largura_original} -> {altura_final}x{largura_final}"
        )

    nome_saida = f"{caminho_entrada.stem}_PP_PL_{PIPELINE_NUMBER}.png"
    caminho_saida = OUTPUT_DIR / nome_saida
    mpimg.imsave(caminho_saida, imagem_final, cmap="gray")
    print(f"Guardado: {caminho_saida.name}")

print("Pipeline 02 concluída.")